# 02 · SONAR Embedding Exploration
Encode/decode quality, cosine similarity structure, boundary scoring, fragility analysis.

**Papers:** LCM ([2412.08821](https://arxiv.org/abs/2412.08821)), SONAR ([2308.11466](https://arxiv.org/abs/2308.11466))

In [ ]:
# pip install sonar-space torch
import torch, numpy as np, matplotlib.pyplot as plt
from scipy.spatial.distance import cosine as cos_dist
print('Note: SONAR downloads ~2 GB of weights on first run.')

## 1. Round-trip encode → decode (AutoBLEU proxy)

In [ ]:
from sonar.inference_pipelines.text import (
    TextToEmbeddingModelPipeline, EmbeddingToTextModelPipeline)
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
embedder = TextToEmbeddingModelPipeline(
    encoder="text_sonar_basic_encoder",
    tokenizer="text_sonar_basic_encoder", device=device)
decoder_sonar = EmbeddingToTextModelPipeline(
    decoder="text_sonar_basic_decoder",
    tokenizer="text_sonar_basic_encoder", device=device)

sentences = [
    "The boundary detector measures cosine dissimilarity between adjacent token representations.",
    "Large concept models reason at the sentence level rather than the token level.",
    "The cat sat on the mat.",
]
embs  = embedder.predict(sentences, source_lang="eng_Latn")
recon = decoder_sonar.predict(embs, target_lang="eng_Latn", max_seq_len=256)
for o, r in zip(sentences, recon):
    print(f"  ORIG : {o}")
    print(f"  RECON: {r}")
    print()

## 2. Inter-sentence cosine similarity matrix

In [ ]:
paragraph = [
    "LLMs apply uniform computation to all tokens.",
    "Language has highly non-uniform information density.",
    "Predictable spans are interspersed with semantically critical transitions.",
    "Standard LLMs waste compute on predictable tokens.",
    "We propose DLCM: a hierarchical next-token prediction framework.",
    "DLCM discovers variable-length concepts end-to-end.",
]
E = embedder.predict(paragraph, source_lang="eng_Latn").cpu().numpy()
n = len(paragraph)
S = np.array([[1 - cos_dist(E[i], E[j]) for j in range(n)] for i in range(n)])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(S, cmap='RdYlBu', vmin=0.5, vmax=1.0)
plt.colorbar(im, ax=ax, label='Cosine Similarity')
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels([f"S{i+1}" for i in range(n)], rotation=45)
ax.set_yticklabels([f"S{i+1}" for i in range(n)])
ax.set_title('SONAR inter-sentence cosine similarity')
plt.tight_layout(); plt.savefig('../data/samples/sonar_similarity.png', dpi=150); plt.show()

## 3. Boundary scoring on SONAR embeddings (DLCM Eq. 6)

In [ ]:
E_t = torch.tensor(E)
E_n = F.normalize(E_t, dim=-1)
cos = (E_n[:-1] * E_n[1:]).sum(-1)
p   = (1 - cos) / 2

print("Sentence-level boundary probabilities:")
for i, pi in enumerate(p.tolist()):
    flag = "  <<< BOUNDARY" if pi > 0.3 else ""
    print(f"  S{i+1}->S{i+2}: p={pi:.4f}{flag}")

plt.figure(figsize=(8, 3))
plt.bar(range(len(p)), p.numpy(),
        color=['#C00000' if pi > 0.5 else '#2E75B6' for pi in p.numpy()])
plt.axhline(0.5, color='black', linestyle='--', label='hard threshold')
plt.xlabel('Position'); plt.ylabel('Boundary prob'); plt.legend()
plt.title('DLCM boundary scoring on SONAR embeddings')
plt.tight_layout(); plt.show()

## 4. Fragility analysis (LCM Section 2.5.2)

In [ ]:
def fragility(text, alpha=0.1, n=6):
    emb   = embedder.predict([text], source_lang="eng_Latn")
    emb_n = F.normalize(emb, dim=-1)
    scores = []
    for _ in range(n):
        eps = torch.randn_like(emb_n)
        pert = ((1-alpha)**0.5 * emb_n + alpha**0.5 * eps)
        pert = F.normalize(pert, dim=-1)
        rec  = decoder_sonar.predict(pert, target_lang="eng_Latn", max_seq_len=128)[0]
        ow   = set(text.lower().split()); rw = set(rec.lower().split())
        scores.append(len(ow & rw) / max(len(ow), 1))
    return float(np.mean(scores))

cases = [
    "The cat sat on the mat.",
    "Backpropagation computes gradients via the chain rule.",
    "John F. Kennedy was assassinated on November 22, 1963.",
]
print("Fragility (higher overlap = more robust, alpha=0.1):")
for t in cases:
    print(f"  {fragility(t):.3f}  {t}")